# 104 職缺資料分析

讀取 `output/104/` 的爬蟲結果，整理薪資欄位後做基本統計與篩選。

- 在 VS Code 中選擇 `/home/node/.venv/bin/python` 作為 kernel（devcontainer 內的虛擬環境，不是工作目錄下的 `.venv`）。
- 輸出內含職缺資料，由 git filter（`scripts/setup-dev-env.sh` 設定）在 `git add` 時移除，不會進版控。

## 1. 載入資料

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

# notebook 的工作目錄是 notebooks/，往上一層是專案根目錄
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output" / "104"

# 要分析的檔名樣式；只看特定一次抓取時改成完整檔名
FILE_PATTERN = "*.json"

# 年薪換算月薪的除數，與 preferences.yaml 的「年薪換算月數」相同意義
ANNUAL_MONTHS = 14

In [ ]:
files = sorted(OUTPUT_DIR.glob(FILE_PATTERN))
frames = []
for path in files:
    with path.open(encoding="utf-8") as f:
        frame = pd.DataFrame(json.load(f))
    frame["來源檔案"] = path.name
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
print(f"[i] 讀取 {len(files)} 個檔案，共 {len(raw)} 筆")
raw.groupby("來源檔案").size().rename("筆數")

不同關鍵字的抓取結果可能有相同職缺，依 `職缺代碼` 去重，保留 `更新日期` 最新的版本；同一天的版本再比檔名中的抓取時間（`_YYYYMMDD_HHMMSS`），保留較晚抓取的。

In [ ]:
# 更新日期是 YYYY-MM-DD 字串，可直接排序；缺值排最前面，不會被優先保留
raw["抓取時間"] = raw["來源檔案"].str.extract(r"_(\d{8}_\d{6})\.json$", expand=False)
df = (
    raw.sort_values(["更新日期", "抓取時間"], na_position="first", kind="stable")
    .drop_duplicates("職缺代碼", keep="last")
    .sort_index()
    .reset_index(drop=True)
)
print(f"[i] 去重後 {len(df)} 筆（移除 {len(raw) - len(df)} 筆重複）")
df.head()

## 2. 整理薪資欄位

`薪資下限`、`薪資上限` 不能直接拿來統計：

- 待遇面議的上下限都是 `0`。
- 「以上」的上限是 `9999999`。
- 月薪、年薪、時薪、論件計酬都放在同樣兩欄，單位不同。

以下新增三欄：

| 欄位 | 說明 |
| :--- | :--- |
| `薪資類型` | `薪資待遇` 的開頭：月薪、年薪、時薪、日薪、論件計酬、待遇面議、其他、未知（`薪資待遇` 為 null） |
| `月薪下限`、`月薪上限` | 只有月薪與年薪有值，年薪除以 `ANNUAL_MONTHS`；無法換算或沒有上限時為 `NaN` |

In [ ]:
NO_UPPER_BOUND = 9999999
SALARY_KINDS = ["月薪", "年薪", "時薪", "日薪", "論件計酬", "待遇面議"]

def salary_kind(text):
    if not isinstance(text, str):
        return "未知"
    return next((kind for kind in SALARY_KINDS if text.startswith(kind)), "其他")

df["薪資類型"] = df["薪資待遇"].map(salary_kind)

low = df["薪資下限"].where(df["薪資下限"] > 0)
high = df["薪資上限"].where((df["薪資上限"] > 0) & (df["薪資上限"] < NO_UPPER_BOUND))
divisor = df["薪資類型"].map({"月薪": 1, "年薪": ANNUAL_MONTHS})  # 其他類型為 NaN，換算結果也是 NaN

# 與評分程式相同，.5 一律進位（不用 round() 的五成雙）
df["月薪下限"] = np.floor(low / divisor + 0.5).astype("Int64")
df["月薪上限"] = np.floor(high / divisor + 0.5).astype("Int64")
df["無上限"] = df["薪資上限"] >= NO_UPPER_BOUND

df["薪資類型"].value_counts().rename("筆數")

檢查：`薪資待遇` 文字中的數字應與 `薪資下限`、`薪資上限` 一致。下表列出不一致的職缺，空表代表資料乾淨。

In [ ]:
import re

def text_matches_numbers(row):
    numbers = [int(n.replace(",", "")) for n in re.findall(r"\d[\d,]*", row["薪資待遇"] or "")]
    if not numbers:
        return row["薪資下限"] == 0 and row["薪資上限"] == 0
    if "以上" in row["薪資待遇"]:
        return numbers[0] == row["薪資下限"] and row["薪資上限"] >= NO_UPPER_BOUND
    return (numbers[0], numbers[-1]) == (row["薪資下限"], row["薪資上限"])

df.loc[~df.apply(text_matches_numbers, axis=1), ["職缺代碼", "薪資待遇", "薪資下限", "薪資上限"]]

## 3. 月薪統計（月薪與年薪換算）

In [ ]:
monthly = df.dropna(subset=["月薪下限"])
print(f"[i] 可換算月薪 {len(monthly)} 筆，其中無上限 {monthly['無上限'].sum()} 筆")
monthly[["月薪下限", "月薪上限"]].describe().round(0)

In [ ]:
# 月薪下限分布，每 10,000 一個區間
bins = range(20000, 110001, 10000)
pd.cut(monthly["月薪下限"].astype(float), bins=bins, right=False).value_counts().sort_index().rename("筆數")

## 4. 產業、地區、技能分布

In [ ]:
df["縣市"] = df["地區"].str[:3]
df["產業類別"].value_counts().head(15).rename("筆數")

In [ ]:
df["縣市"].value_counts().rename("筆數")

In [ ]:
# 電腦專長、特色標籤是以「, 」分隔的字串
def split_counts(column, top=20, normalize=None):
    items = df[column].fillna("").str.split(", ").explode().loc[lambda s: s != ""]
    if normalize:
        items = items.map(normalize)
    return items.value_counts().head(top).rename("筆數")

split_counts("電腦專長")

In [ ]:
# 「距捷運○○站約N公尺」每筆都不同，合併成「近捷運」
split_counts("特色標籤", normalize=lambda tag: "近捷運" if tag.startswith("距捷運") else tag)

In [ ]:
# 各產業的月薪下限中位數（只計可換算的職缺）
(
    df.dropna(subset=["月薪下限"])
    .groupby("產業類別")["月薪下限"]
    .agg(筆數="count", 中位數="median")
    .sort_values("筆數", ascending=False)
    .head(15)
)

## 5. 篩選職缺

調整下方條件後執行。`MIN_MONTHLY` 為 `None` 時不篩薪資，薪資未知（面議、時薪等）的職缺由 `INCLUDE_UNKNOWN_SALARY` 決定是否保留。

In [ ]:
MIN_MONTHLY = 50000
INCLUDE_UNKNOWN_SALARY = True
CITIES = ["台北市", "新北市"]      # 空清單代表不篩
SKILL_KEYWORD = "Python"          # 空字串代表不篩，比對電腦專長與職缺名稱（不分大小寫）

mask = pd.Series(True, index=df.index)
if MIN_MONTHLY is not None:
    # 上限達標即可（無上限也算），與評分的「區間涵蓋期望」同義
    reaches = (df["月薪上限"] >= MIN_MONTHLY).fillna(False) | (df["無上限"] & df["月薪下限"].notna())
    unknown = df["月薪下限"].isna()
    mask &= reaches | (unknown & INCLUDE_UNKNOWN_SALARY)
if CITIES:
    mask &= df["縣市"].isin(CITIES)
if SKILL_KEYWORD:
    text = df["電腦專長"].fillna("") + " " + df["職缺名稱"].fillna("")
    mask &= text.str.contains(SKILL_KEYWORD, case=False, regex=False)

picked = df.loc[mask, ["職缺名稱", "公司名稱", "縣市", "薪資待遇", "月薪下限", "月薪上限", "更新日期", "應徵人數", "職缺連結"]]
print(f"[i] 符合條件 {len(picked)} 筆")
picked.sort_values(["月薪下限", "更新日期"], ascending=False, na_position="last")